[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Loading Strategies


## What you will be able to do

Count the queries a piece of code sends, and find the N plus 1 queries hidden in an ordinary loop
over related objects. Load related objects with the query instead, with `selectinload` and
`joinedload`, choose between them, and chain them along a path of relationships. Make any forgotten
lazy load an error with `raiseload`, and recognize the errors from a joined collection without
`unique()`, a lazy load that was forbidden, one attempted after the session closed, and an eager
load that stopped a step short.


## The idea

### The problem

The registrar prints the class lists for Spring 2026: ten sections, and for each one its course's
code and title, the number of students, and their names. The code is the obvious loop, which the
**Relationships** notebook made easy to write: for every section, `section.course.title`, and for
every enrollment, `enrollment.student.name`. It prints the right lists. It also sends 46 queries:
one for the sections, and then one for every section's course, one for every section's enrollments,
and one for every student the first time the loop meets them.

Ten sections and 46 queries is quick on a laptop. A college with four hundred sections sends several
thousand, every one of them small, and together slow, and against a server across a network, much
slower. Nothing in the loop looks like a query, which is why it goes unnoticed until a page that
lists many objects turns out to be slow.

### What a loading strategy is

> A **loading strategy** decides when a relationship's objects are loaded. **Lazy loading**, the
> default, sends a `SELECT` the first time the attribute of one object is read, which is where the
> **N plus 1 queries** come from: one query for N objects, then one more for each of them.
> **Eager loading** loads the related objects along with the query. **`selectinload`** sends one
> more `SELECT` for a relationship, with the keys of every parent in an `IN` list, and
> **`joinedload`** adds a join to the query itself, so the related rows arrive in the same rows as
> their parents. **`raiseload`** makes a lazy load an error, so a missing eager load cannot go
> unnoticed. A strategy is chosen for one query with `.options()`, or for every query with
> `relationship(lazy=...)`.

### Why it works that way

- **Lazy loading costs nothing until it is used**, and one query every time it is. For one object
  that is exactly right, and in a loop it is one query for every pass.
- **`selectinload` costs one query for every relationship**, however many parents there are, since
  all of their keys go into one `IN` list.
- **`joinedload` costs no query, and multiplies rows.** A join repeats the parent's columns in every
  row of its children, which is free for a many to one relationship, one row either way, and for a
  collection means the same parent in many rows, which `unique()` folds back into one.
- **Strategies follow a path.** `selectinload(Section.enrollments).joinedload(Enrollment.student)`
  loads the enrollments, and every enrollment's student, and any step left out is lazy again.
- **Only an open session can load.** A collection loaded before the session closed stays readable
  after it; one that was never loaded cannot be loaded later.

### Where this shows up

Every page or API response that lists objects with their related data meets this, and the usual fix
is the one this notebook teaches. Django's `select_related` and `prefetch_related` are the same two
ideas as `joinedload` and `selectinload`. The **Async SQLAlchemy** notebook needs eager loading
outright, since a lazy load there is an error, and the **Joins and Aggregates** notebook joins for
filtering, which is a different job from joining for loading.

### What this notebook covers

- Counting the statements a block of code sends
- The N plus 1 queries in an ordinary loop
- `selectinload`: one query for every relationship
- `joinedload`: the related rows in the query itself, and `unique()`
- A `LIMIT` with a joined collection
- `raiseload`: a lazy load as an error
- Which strategy to use when
- Class lists for a term in two queries, finished
- Four errors, from a joined collection without `unique()` to an eager load a step short

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import ForeignKey, create_engine, event, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship
from sqlalchemy.orm import selectinload


class Base(DeclarativeBase):
    pass


class Course(Base):
    __tablename__ = "courses"
    id: Mapped[int] = mapped_column(primary_key=True)
    sections: Mapped[list["Section"]] = relationship()


class Section(Base):
    __tablename__ = "sections"
    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))


engine = create_engine("sqlite://")
Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add_all([Course(sections=[Section(), Section()]) for _ in range(5)])
    session.commit()

sent = []
event.listen(engine, "before_cursor_execute", lambda *args: sent.append(args[2]))
for options in ([], [selectinload(Course.sections)]):
    sent.clear()
    with Session(engine) as session:
        courses = session.scalars(select(Course).options(*options)).all()
        total = sum(len(course.sections) for course in courses)
    print(len(sent), "queries for", total, "sections")
```

```
6 queries for 10 sections
2 queries for 10 sections
```

The same loop over five courses and their sections, twice. Left lazy, it sent one query for the
courses and one for every course's sections. With `selectinload`, it sent two, whatever the number
of courses. The listener on `before_cursor_execute` records every statement the engine sends, which
is how the counts were made.


## Setup

Fourteen imports, the college built from its classes, and a helper for error messages.

- `sqlalchemy` is the library itself, and the cell prints its version
- `selectinload`, `joinedload` and `raiseload`, from `sqlalchemy.orm`, choose how a query loads
  relationships, with `relationship` and the rest of what the classes need, `Session` and
  `sessionmaker`
- `event`, from `sqlalchemy`, listens for every statement an engine sends, and `contextmanager`, from
  `contextlib`, turns a counting function into a `with` block
- `DetachedInstanceError`, from `sqlalchemy.orm.exc`, is the error a lazy load after a close raises,
  and `re` takes the memory address out of its message; `InvalidRequestError`, from `sqlalchemy.exc`,
  is the error a forbidden lazy load raises
- `select`, `func` and `insert`, `create_engine`, and what the classes need, `MetaData`, `String`,
  `ForeignKey` and the constraints
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what the `Date` columns take and return
- `logging` carries the SQL an engine logs to `PrintStatements`
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

Setup builds the college from the classes of the **Relationships** notebook, and `without_address`
is the helper from **The Identity Map** notebook, which replaces the memory address in an error's
message.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import logging
import re
import shutil
from contextlib import contextmanager
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, create_engine, event, func,
                        insert, select)
from sqlalchemy.orm import (DeclarativeBase, Mapped, Session, joinedload, mapped_column, raiseload, relationship,
                            selectinload, sessionmaker)
from sqlalchemy.exc import InvalidRequestError
from sqlalchemy.orm.exc import DetachedInstanceError
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))

SessionLocal = sessionmaker(engine)


def without_address(error):
    """An error's message with every memory address replaced, since the addresses change on every run."""
    return re.sub(r"0x[0-9a-f]+", "0x...", str(error))


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


## Worked examples

### Counting the statements a block of code sends

`echo` shows every statement, which is too much to read in a loop that sends dozens. An event
listener on `before_cursor_execute` runs for every statement the engine sends, so a listener that
adds one to a counter, attached for the length of a `with` block, counts them:


In [2]:
@contextmanager
def counting(engine):
    """Count the statements an engine sends while the block runs, in a dictionary the block can read."""
    counter = {"statements": 0}

    def count(conn, cursor, statement, parameters, context, executemany):
        counter["statements"] += 1

    event.listen(engine, "before_cursor_execute", count)
    try:
        yield counter
    finally:
        event.remove(engine, "before_cursor_execute", count)


with SessionLocal() as session, counting(engine) as sent:
    chloe = session.get(Student, 3)
    first_course = chloe.enrollments[0].section.course
    print(first_course, "| statements:", sent["statements"])


Course('CHE-110', 4) | statements: 4


Four statements to reach one course: Chloe Martin, her enrollments, the first enrollment's section,
and that section's course, one lazy load for every step. `event.listen` attaches the function and
`event.remove` takes it off again, so the counter sees only the block's statements.

### The N plus 1 queries in an ordinary loop

The class lists for Spring 2026, written the obvious way. `class_list_lines` reads every section's
course and every enrollment's student, and the query loads the sections and nothing more:


In [3]:
SPRING_SECTIONS = select(Section).where(Section.term_id == 4).order_by(Section.id)


def class_list_lines(sections):
    """One line for every section: its course, how many students it has, and the first three of them."""
    lines = []
    for section in sections:
        names = [enrollment.student.name for enrollment in section.enrollments]
        lines.append(f"{section.course.code}  {section.course.title:<27} {len(names)} students: {', '.join(names[:3])}")
    return lines


with SessionLocal() as session, counting(engine) as sent:
    lines = class_list_lines(session.scalars(SPRING_SECTIONS))
print("\n".join(lines[:3]))
print("statements:", sent["statements"])


BIO-101  Introduction to Biology     8 students: Ana Reyes, Daniel Kim, Grace Lin
CHE-110  General Chemistry           8 students: Ben Okafor, Elena Petrova, Hassan Ali
MAT-120  Calculus I                  7 students: Chloe Martin, Felix Wagner, Isabel Costa
statements: 46


Forty-six statements for ten short lines. One loaded the ten sections. Then every section sent one
for its course and one for its enrollments, twenty more, and every student sent one the first time
the loop reached them, twenty-five more. A student met a second time came from the identity map with
no query, which is the only thing that kept the count from being higher.

### selectinload: one query for every relationship

`.options()` tells one query how to load. `selectinload(Section.enrollments)` loads every section's
enrollments in one `SELECT`, and `.selectinload(Enrollment.student)` after it does the same for the
students of those enrollments. With `echo` on, the statements are few enough to read:


In [4]:
SELECTIN = SPRING_SECTIONS.options(
    selectinload(Section.course),
    selectinload(Section.enrollments).selectinload(Enrollment.student),
)

engine.echo = True
with SessionLocal() as session, counting(engine) as sent:
    lines = class_list_lines(session.scalars(SELECTIN))
engine.echo = False
print(lines[0])
print("statements:", sent["statements"])


    BEGIN (implicit)
    SELECT sections.id, sections.course_id, sections.term_id, sections.capacity
    FROM sections
    WHERE sections.term_id = ? ORDER BY sections.id
    values: (4,)
    SELECT courses.id AS courses_id, courses.code AS courses_code, courses.title AS courses_title, courses.department AS courses_department, courses.credits AS courses_credits
    FROM courses
    WHERE courses.id IN (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    values: (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
    SELECT enrollments.section_id AS enrollments_section_id, enrollments.student_id AS enrollments_student_id, enrollments.status AS enrollments_status, enrollments.grade AS enrollments_grade
    FROM enrollments
    WHERE enrollments.section_id IN (?, ?, ?, ?, ?, ?, ?, ?, ?, ?) ORDER BY enrollments.student_id
    values: (31, 32, 33, 34, 35, 36, 37, 38, 39, 40)
    SELECT students.id AS students_id, students.name AS students_name, students.email AS students_email, students.program AS students_program, students.sta

Four statements: the sections, then their courses, their enrollments and the enrollments' students,
each loaded for every parent at once with an `IN` list of the keys the previous query found. The
loop itself sent nothing, since everything it read was already there. Four is the number for this
path of relationships whether there are ten sections or four hundred; for very long lists,
`selectinload` splits the keys into batches.

### joinedload: the related rows in the query itself, and unique()

`joinedload` adds a `LEFT OUTER JOIN` to the query, so a section's course arrives in the section's
own row. For a many to one relationship that costs nothing, since every section has exactly one
course:


In [5]:
JOINED_COURSE = SPRING_SECTIONS.options(joinedload(Section.course))
print(" ".join(str(JOINED_COURSE.compile(engine)).split()))

with SessionLocal() as session, counting(engine) as sent:
    sections = session.scalars(JOINED_COURSE).all()
    titles = [section.course.title for section in sections]
print(len(sections), "sections,", len(set(titles)), "courses | statements:", sent["statements"])


SELECT sections.id, sections.course_id, sections.term_id, sections.capacity, courses_1.id AS id_1, courses_1.code, courses_1.title, courses_1.department, courses_1.credits FROM sections LEFT OUTER JOIN courses AS courses_1 ON courses_1.id = sections.course_id WHERE sections.term_id = ? ORDER BY sections.id
10 sections, 10 courses | statements: 1


One statement for the sections and their courses together, with the course's columns under the alias
`courses_1`, so that the join cannot be confused with a join the query makes for its own reasons.
For a collection, a join repeats the section in a row for every enrollment. The rows have to be
folded back into one object a section, and `unique()` on the result does that:


In [6]:
JOINED_ENROLLMENTS = SPRING_SECTIONS.options(joinedload(Section.enrollments))

SAME_JOIN = select(func.count()).select_from(Section).join(Section.enrollments).where(Section.term_id == 4)

with SessionLocal() as session:
    rows = session.scalar(SAME_JOIN)                          # the rows a join of sections to enrollments returns
    sections = session.scalars(JOINED_ENROLLMENTS).unique().all()
print("rows the join returns:", rows, "| sections after unique():", len(sections))


rows the join returns: 75 | sections after unique(): 10


Seventy-five rows came back, one for every enrollment in Spring 2026, and `unique()` turned them
into ten sections, each holding its enrollments. Without `unique()` SQLAlchemy refuses to hand over
the result at all, which is the first of the Common errors. The count came from a query with the
same join written out, since the join that `joinedload` adds belongs to the loading, and a count
cannot see it.

### A LIMIT with a joined collection

A `LIMIT` counts rows, and a joined collection makes several rows of one parent, so a query for the
first three students, joined to their enrollments, could stop partway through the first student's
enrollments. SQLAlchemy prevents that, which its SQL shows:


In [7]:
FIRST_THREE = select(Student).options(joinedload(Student.enrollments)).order_by(Student.id).limit(3)
print(" ".join(str(FIRST_THREE.compile(engine)).split()))

with SessionLocal() as session:
    for student in session.scalars(FIRST_THREE).unique():
        print(f"{student.name:<13} {len(student.enrollments)} enrollments")


SELECT anon_1.id, anon_1.name, anon_1.email, anon_1.program, anon_1.started_on, enrollments_1.student_id, enrollments_1.section_id, enrollments_1.status, enrollments_1.grade FROM (SELECT students.id AS id, students.name AS name, students.email AS email, students.program AS program, students.started_on AS started_on FROM students ORDER BY students.id LIMIT ? OFFSET ?) AS anon_1 LEFT OUTER JOIN enrollments AS enrollments_1 ON anon_1.id = enrollments_1.student_id ORDER BY anon_1.id, enrollments_1.section_id
Ana Reyes     12 enrollments
Ben Okafor    9 enrollments
Chloe Martin  6 enrollments


The `LIMIT` went inside a subquery of students, `anon_1`, and the join to the enrollments was made
outside it, so the limit counted students, three of them, and every student came with all of their
enrollments. This is SQLAlchemy 2.0's behavior with `joinedload`: a join the program writes itself,
as the **Joins and Aggregates** notebook does, has no such protection.

### raiseload: a lazy load as an error

The N plus 1 queries went unnoticed because a lazy load looks like an attribute. `raiseload` makes
it an error instead, for one relationship, or for every relationship the query did not load, with
`"*"`:


In [8]:
STRICT = SPRING_SECTIONS.options(selectinload(Section.enrollments), raiseload("*"))

with SessionLocal() as session:
    section = session.scalars(STRICT).first()
    print(section, "has", len(section.enrollments), "enrollments")
    try:
        section.course
    except InvalidRequestError as error:
        print("refused:", error)


Section(31) has 8 enrollments
refused: 'Section.course' is not available due to lazy='raise'


The enrollments had been loaded, so reading them was fine, and the course had not, so reading it
raised instead of sending a query. With `raiseload("*")` on a query, every relationship the code
reads has to be listed in `.options()`, which turns a forgotten eager load into an error in testing
rather than a slow page later.

### Which strategy to use when

| Use | When | Why |
|---|---|---|
| lazy loading, the default | one object, or a relationship read now and then | no cost until the attribute is read |
| `selectinload` | a collection, loaded for many parents | one extra query, whatever the number of parents |
| `joinedload` | a many to one relationship, such as a section's course | no extra query, and no repeated rows |
| `joinedload` on a collection, with `unique()` | a collection whose parents are few, when one query matters | one query, at the price of repeated rows |
| `raiseload("*")` | a query that must not trigger any lazy load, in tests or in async code | a forgotten eager load becomes an error |
| `relationship(lazy="selectin")` | a relationship that every query needs | changes the default for every query, which `.options()` can still override |

The default is lazy loading for a single object, and `selectinload` for collections and `joinedload`
for many to one relationships in anything that loops, with `raiseload("*")` to prove nothing was
missed.

### Class lists for a term in two queries, finished

The pieces of this notebook in one function. `class_lists` loads a term's sections with their
courses joined in, their enrollments selected in, and every enrollment's student joined into that
second query, then forbids any other lazy load, and uses the same `class_list_lines` as the loop
that sent forty-six:


In [9]:
def class_lists(session, term):
    """The class lists of a term, loaded in two queries, with every lazy load turned into an error."""
    query = (
        select(Section)
        .join(Section.term)
        .where(Term.name == term)
        .order_by(Section.id)
        .options(
            joinedload(Section.course),
            selectinload(Section.enrollments).joinedload(Enrollment.student),
            raiseload("*"),
        )
    )
    return class_list_lines(session.scalars(query))



with SessionLocal() as session, counting(engine) as sent:
    lines = class_lists(session, "Spring 2026")
print("\n".join(lines))
print("statements:", sent["statements"])


BIO-101  Introduction to Biology     8 students: Ana Reyes, Daniel Kim, Grace Lin
CHE-110  General Chemistry           8 students: Ben Okafor, Elena Petrova, Hassan Ali
MAT-120  Calculus I                  7 students: Chloe Martin, Felix Wagner, Isabel Costa
MAT-121  Calculus II                 7 students: Daniel Kim, Grace Lin, Jonas Berg
CSC-101  Programming I               8 students: Ana Reyes, Elena Petrova, Hassan Ali
CSC-201  Data Structures             7 students: Ben Okafor, Felix Wagner, Isabel Costa
ENG-105  Composition                 7 students: Chloe Martin, Grace Lin, Jonas Berg
HIS-110  World History               8 students: Ana Reyes, Daniel Kim, Hassan Ali
PSY-101  Introduction to Psychology  8 students: Ben Okafor, Elena Petrova, Isabel Costa
STA-200  Statistics                  7 students: Chloe Martin, Felix Wagner, Jonas Berg
statements: 2


Two statements: the sections with their courses in one, and the enrollments with their students in
the other. The function that prints the lists did not change at all, which is the point of loading
strategies: the loop stays the natural one, and the query decides what is loaded. `raiseload("*")`
promises that nothing else was, since any other lazy load would have raised.

### Where each part came from

| In `class_lists` | What it relies on | The section that showed it |
|---|---|---|
| `joinedload(Section.course)` | a many to one relationship in the query's own rows | joinedload: the related rows in the query itself, and unique() |
| `selectinload(Section.enrollments)` | a collection in one more query | selectinload: one query for every relationship |
| `.joinedload(Enrollment.student)` | a strategy for the next step of the path | selectinload: one query for every relationship |
| `raiseload("*")` | any other lazy load refused | raiseload: a lazy load as an error |
| the count of two | an event listener counting statements | Counting the statements a block of code sends |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/13-loading-strategies-solutions.ipynb).

**1.** Count the statements sent by a loop that prints every History student's number of enrollments,
lazily and then with `selectinload`.


In [10]:
# your code here


**2.** Load Chloe Martin's transcript, every enrollment with its section's course and term, in as few
statements as you can, and print the count.


In [11]:
# your code here


**3.** Print the SQL of a query for Fall 2025's sections with `joinedload(Section.course)` and
`joinedload(Section.term)`, and say how many joins it has.


In [12]:
# your code here


**4.** Load every course with its sections, and every section with its term, using `selectinload` for
both steps, and print the number of statements and of sections.


In [13]:
# your code here


**5.** Use `raiseload("*")` to find which relationships `class_list_lines` reads: start with no eager
loads, and add one at a time until it runs.


In [14]:
# your code here


**6.** Load the sections of Spring 2026 with their enrollments, close the session, and print every
section's number of students after the close.


In [15]:
# your code here


## Common errors

### sqlalchemy.exc.InvalidRequestError: The unique() method must be invoked on this Result, as it contains results that include joined eager loads against collections


In [16]:
with SessionLocal() as session:
    sections = session.scalars(SPRING_SECTIONS.options(joinedload(Section.enrollments))).all()


InvalidRequestError: The unique() method must be invoked on this Result, as it contains results that include joined eager loads against collections

A joined collection returns every section once for every enrollment it has, and a result that handed
those rows over would give the same section seventy-five times over. SQLAlchemy refuses rather than
fold them without being asked. Call `unique()` on the result, or load the collection with
`selectinload`, which never repeats a parent:


In [17]:
with SessionLocal() as session:
    joined = session.scalars(SPRING_SECTIONS.options(joinedload(Section.enrollments))).unique().all()
    selected = session.scalars(SPRING_SECTIONS.options(selectinload(Section.enrollments))).all()
print(len(joined), "sections either way:", len(selected))


10 sections either way: 10


### sqlalchemy.exc.InvalidRequestError: 'Section.course' is not available due to lazy='raise'


In [18]:
with SessionLocal() as session:
    strict = SPRING_SECTIONS.options(selectinload(Section.enrollments).selectinload(Enrollment.student), raiseload("*"))
    lines = class_list_lines(session.scalars(strict))


InvalidRequestError: 'Section.course' is not available due to lazy='raise'

`raiseload("*")` did its job: the query loaded the enrollments and their students, and
`class_list_lines` also reads every section's course, which the query never mentioned. The message
names the relationship to add to `.options()`, `joinedload(Section.course)` here, as `class_lists`
has.

### sqlalchemy.orm.exc.DetachedInstanceError: Parent instance <Section at 0x...> is not bound to a Session; lazy load operation of attribute 'enrollments' cannot proceed


In [19]:
with SessionLocal() as session:
    section = session.get(Section, 31)                       # the enrollments were never loaded

try:
    print(len(section.enrollments))
except DetachedInstanceError as error:
    print("DetachedInstanceError:", without_address(error))


DetachedInstanceError: Parent instance <Section at 0x...> is not bound to a Session; lazy load operation of attribute 'enrollments' cannot proceed (Background on this error at: https://sqlalche.me/e/20/bhk3)


The session that loaded the section closed before anything read its enrollments, and a detached
object cannot send the lazy load that reading them needs. The message names the object's memory
address, which the cell replaced before printing, since it changes on every run. Load what the code
after the block will read while the session is open, with an eager load:


In [20]:
with SessionLocal() as session:
    section = session.scalars(select(Section).where(Section.id == 31).options(selectinload(Section.enrollments))).one()

print(section, "has", len(section.enrollments), "enrollments, read after the session closed")


Section(31) has 8 enrollments, read after the session closed


### No error, and twenty-five queries left: an eager load that stopped a step short


In [21]:
SHORT = SPRING_SECTIONS.options(joinedload(Section.course), selectinload(Section.enrollments))

with SessionLocal() as session, counting(engine) as sent:
    lines = class_list_lines(session.scalars(SHORT))
print("statements:", sent["statements"])


statements: 27


The sections, their courses and their enrollments were loaded eagerly, and every enrollment's
student was not, so the loop still sent one query for every student, twenty-seven statements in all.
An eager load covers one step of a path, and the next step is lazy again unless it is chained. Chain
it, and add `raiseload("*")` so that the next step left out is an error instead of a count:


In [22]:
FULL = SPRING_SECTIONS.options(joinedload(Section.course), selectinload(Section.enrollments).joinedload(Enrollment.student),
                               raiseload("*"))

with SessionLocal() as session, counting(engine) as sent:
    lines = class_list_lines(session.scalars(FULL))
print("statements:", sent["statements"])


statements: 2


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [23]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- Lazy loading sends a query every time a relationship of one object is read for the first time, so a
  loop over related objects sends N plus 1 queries; an event listener counts them.
- `selectinload` loads a relationship for every parent in one more query, and suits collections.
- `joinedload` loads it in the query's own rows, suits many to one relationships, and needs
  `unique()` on a collection; with a `LIMIT`, SQLAlchemy keeps the limit counting parents.
- Strategies chain along a path, and every step left out is lazy again.
- `raiseload("*")` turns any lazy load the query did not plan for into an error, and an unloaded
  relationship cannot be loaded once its session has closed.


## What is next

The **Joins and Aggregates** notebook joins for filtering and counting rather than for loading:
`join`, `aliased`, `group_by`, `any` and `has`, and the warning that you built a cartesian product.


---

&#8592; **Previous:** [Many to Many](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/12-many-to-many.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
